# Upstage (2026 업데이트판)

Upstage는 LLM(Solar)과 문서 AI 분야에 특화된 국내 AI 기업입니다.

**API 키 발급**: [Upstage Console](https://console.upstage.ai/api-keys) → `.env`에 `UPSTAGE_API_KEY=...`

### 원본 대비 변경 사항
| 항목 | 원본 | 현재 권장 |
|---|---|---|
| 모델 객체 | 쿼리용·문서용 객체를 따로 생성 (`-query`, `-passage` 접미사) | **객체 하나**: `UpstageEmbeddings(model="solar-embedding-1-large")` |
| 쿼리/문서 구분 | 사용자가 직접 객체를 골라 호출 | `embed_query()`는 쿼리 모델, `embed_documents()`는 문서 모델을 **자동 사용** |
| 유사도 | 내적 | 정규화된 코사인 유사도 헬퍼 |

객체가 하나이므로 벡터 저장소(`vectorstore.from_documents(..., embedding=...)`)나 retriever에 그대로 넘길 수 있습니다. 원본처럼 객체를 둘로 나누면, 벡터 저장소가 검색할 때도 문서용 모델로 쿼리를 임베딩하는 실수가 생기기 쉽습니다.

LangChain 공식 문서도 모델명에 `-query`, `-passage` 접미사를 붙이지 말라고 안내합니다.

In [ ]:
%pip install -qU langchain-upstage python-dotenv numpy

## 환경 설정

- `.env` 파일의 API 키를 `python-dotenv`로 불러옵니다.
- **변경점**: 책에서 사용한 `langchain_teddynote.logging.langsmith()`는 서드파티 헬퍼입니다. 현재 LangSmith 공식 방식은 환경 변수(`LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`)만 설정하는 것이며, 별도 패키지가 필요 없습니다.
- 참고: 임베딩 호출(`embed_query`, `embed_documents`)은 Runnable이 아니어서 LangSmith에 트레이스가 남지 않습니다. 이 챕터에서는 없어도 되는 설정이지만, 이후 체인/에이전트 실습과 형태를 맞추기 위해 둡니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 파일의 키를 환경 변수로 로드

# LangSmith 추적 (LANGSMITH_API_KEY 는 .env 에 넣어 둡니다)
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "CH08-Embeddings")

In [ ]:
texts = [
    "안녕, 만나서 반가워.",
    "LangChain simplifies the process of building applications with large language models",
    "랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. ",
    "LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.",
    "Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.",
]

**지원 모델 확인**: [Upstage Console](https://console.upstage.ai/) 의 문서(Docs) → Embeddings 항목

| Model | Context Length | 설명 |
|---|---|---|
| solar-embedding-1-large-query | 4000 | 사용자 질문 임베딩에 최적화 |
| solar-embedding-1-large-passage | 4000 | 검색 대상 문서 임베딩에 최적화 |

두 모델은 같은 벡터 공간을 공유하므로 쿼리 벡터와 문서 벡터를 직접 비교할 수 있습니다. LangChain에서는 공통 이름 `solar-embedding-1-large`만 지정하면 됩니다.

In [ ]:
from langchain_upstage import UpstageEmbeddings

# 하나의 객체로 쿼리/문서 모두 처리
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

`Query`를 임베딩합니다. 내부적으로 `solar-embedding-1-large-query`가 사용됩니다.

In [ ]:
query = "LangChain 에 대해서 상세히 알려주세요."

embedded_query = embeddings.embed_query(query)
len(embedded_query)  # 임베딩 차원

문서를 임베딩합니다. 내부적으로 `solar-embedding-1-large-passage`가 사용됩니다.

In [ ]:
embedded_documents = embeddings.embed_documents(texts)

유사도 계산 결과를 출력합니다.

**변경점**: 원본은 `query @ documents.T`(내적)만 사용했습니다. 내적은 벡터가 **정규화되어 있을 때만** 코사인 유사도와 같습니다. 모델마다 정규화 여부가 다르므로, 아래처럼 명시적으로 정규화한 뒤 코사인 유사도를 계산하는 헬퍼를 쓰는 편이 안전합니다.

In [ ]:
import numpy as np


def cosine_scores(query_vec, doc_vecs):
    q = np.asarray(query_vec, dtype=np.float32)
    d = np.asarray(doc_vecs, dtype=np.float32)
    q = q / np.linalg.norm(q)
    d = d / np.linalg.norm(d, axis=1, keepdims=True)
    return d @ q


def print_ranking(query, query_vec, doc_vecs, docs):
    scores = cosine_scores(query_vec, doc_vecs)
    print(f"[Query] {query}\n" + "=" * 40)
    for rank, idx in enumerate(scores.argsort()[::-1]):
        print(f"[{rank}] 유사도: {scores[idx]:.3f} | {docs[idx]}\n")

In [ ]:
print_ranking(query, embedded_query, embedded_documents, texts)